# **Predicting Unpaid Orders in an Online Shop**

## Setup:

In [ ]:
import pandas as pd
import numpy as np
import numpy as np
train_dataset_path = 'train.csv'
test_dataset_path = 'test.csv'

train_raw_df = pd.read_csv(train_dataset_path, delimiter = ',', index_col=0)
test_raw_df = pd.read_csv(test_dataset_path, delimiter = ',', index_col=0)

# random state for all prediction
seed = 13
# splits for cross validation
n_splits = 5

def reload_train_df():
  return train_raw_df.copy()

def reload_test_df():
  return test_raw_df.copy()

# Raw Data

In [ ]:
df = reload_train_df()
pd.set_option("display.max_columns", None)
df.head()

,O_EMAIL,O_TELEPHONE,FLAG_DIIDENTICAL,FLAG_NEWSLETTER,CHK_LADR,CHK_RADR,CHK_KTO,CHK_CARD,CHK_COOKIE,CHK_IP,FAIL_LPLZ,FAIL_LORT,FAIL_LPLZORTMATCH,FAIL_RPLZ,FAIL_RORT,FAIL_RPLZORTMATCH,NEW_CUSTOMER,P_METHOD,P_CARD_TYPE,DEL_DAY,DEL_AMOUNT,DEL_VALUE,SESSION_TIME,DEL_AMOUNT_TOTAL,DEL_VALUE_TOTAL,REM_CUR,REM_MAX,ANUMBER_01,ANUMBER_02,ANUMBER_03,ANUMBER_04,ANUMBER_05,TARGET_FRAUD
Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Invoice,NaN,Sa,1,9.8,8,1,33.47,1.0,2.0,181,NaN,NaN,NaN,NaN,False
1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Credit Card,Visa,Do,1,62.0,9,1,16.64,0.0,1.0,51,NaN,NaN,NaN,NaN,False
2,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,Invoice,NaN,Fr,1,8.5,5,0,0.00,NaN,NaN,468,NaN,NaN,NaN,NaN,False
3,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,Invoice,NaN,Fr,1,10.8,5,0,0.00,NaN,NaN,171,NaN,NaN,NaN,NaN,False
4,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,Invoice,NaN,Di,1,57.8,22,0,0.00,NaN,NaN,290,NaN,NaN,NaN,NaN,False


# **Feature Engineering:**

## Encoding categorical data:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

def encode_categories(dataframe, columns):

    encoder = OneHotEncoder(sparse_output=False)
    encoded_cols = encoder.fit_transform(dataframe[columns])
    column_names = encoder.get_feature_names_out(columns)
    encoded_df = pd.DataFrame(encoded_cols, columns=column_names, index=dataframe.index)
    return pd.concat([dataframe.drop(columns=columns), encoded_df], axis=1)

# I tried to manually onehot encode the article columns, but it just decreased the accuracy
def encode_articles(dataframe, article_columns):

    melted = dataframe[article_columns].melt(value_name="ARTICLE", ignore_index=False).dropna()
    melted["ARTICLE"] = melted["ARTICLE"].astype(int)
    ohe = pd.get_dummies(melted["ARTICLE"], prefix="ARTICLE").groupby(melted.index).max()

    return dataframe.drop(columns=article_columns).join(ohe, how="left")

## Adding new Features:

In [ ]:
def add_avg_item_cost(dataframe, col_name_value, col_name_amount, new_column_name):

    dataframe[new_column_name] = dataframe[col_name_value] / dataframe[col_name_amount]
    dataframe[new_column_name] = dataframe[new_column_name].fillna(0)
    return dataframe


def add_features(dataframe):

    '''
    This feature indicates whether an order was made on a weekend. I assume that genuine customers tend to shop on weekends,
    whereas scammers who might commit fraud as a profession are more likely to operate on weekdays.
    '''
    dataframe['WEEKEND'] = dataframe[['DEL_DAY_Sa', 'DEL_DAY_So']].any(axis=1)


    '''
    This feature represents the average cost per item for the current order. I assume having this information can help
    differentiate orders by providing a detail that was previously unavailable to the model.
    '''
    dataframe = add_avg_item_cost(dataframe, 'DEL_VALUE', 'DEL_AMOUNT', 'AVG_ITEM_COST')


    '''
    This feature represents the average cost per item across all previous orders. I assume that having this information
    can show unusual behavior which can be an indicator for fraud.
    '''
    dataframe = add_avg_item_cost(dataframe, 'DEL_VALUE_TOTAL', 'DEL_AMOUNT_TOTAL', 'AVG_ITEM_COST_USUALLY')


    '''
    This feature represents the absolute difference between a customer's usual average item price and the average item price of
    the current order. I assume that having a high variance to the usual item price, indicates abnormal behavior,
    which could be a sign of fraud.
    '''
    dataframe['DISTANCE_USUAL_COST'] = dataframe.apply(lambda row: abs(row['AVG_ITEM_COST'] - row['AVG_ITEM_COST_USUALLY']) if row['AVG_ITEM_COST_USUALLY'] != 0 else 0, axis=1)


    '''
    This feature indicates whether an identity has done an order in the past three days. Even though the individual
    indicators already exist, aggregating them into a single binary flag increased accuracy
    '''
    dataframe['RECENTLY_ORDERED'] = dataframe[['CHK_LADR', 'CHK_RADR', 'CHK_KTO', 'CHK_CARD', 'CHK_COOKIE', 'CHK_IP']].any(axis=1)

    return dataframe


## Handling NaN:

In [ ]:
def handle_nan(dataframe):
    dataframe['REM_CUR'] = dataframe['REM_CUR'].fillna(0)
    dataframe['REM_MAX'] = dataframe['REM_MAX'].fillna(0)
    return dataframe

## Result of Feature Engineering:

In [ ]:
from sklearn.model_selection import train_test_split

# Reasoning for dropping Features:
## Article numbers seemed pretty weird when encoding and just reduced accuracy
## P_CARD_TYPE_nan is basicly P_METHOD_Invoice
## P_CARD_TYPE_Kundenkarte is basicly 'P_METHOD_Customer Card'
## P_METHOD_Credit Card is already covered by the specific card types. Especially american vs euro

def preprocess_df(dataframe):

  dataframe = encode_categories(dataframe, ['P_METHOD', 'P_CARD_TYPE', 'DEL_DAY'])
  dataframe = add_features(dataframe)
  dataframe = handle_nan(dataframe)
  dataframe = dataframe.drop(columns=['ANUMBER_01',	'ANUMBER_02',	'ANUMBER_03',	'ANUMBER_04',	'ANUMBER_05', 'P_CARD_TYPE_nan', 'P_CARD_TYPE_Kundenkarte', 'P_METHOD_Credit Card'])
  return dataframe


# variables for use in models
df = preprocess_df(reload_train_df())
xx = df.drop(columns=['TARGET_FRAUD'])
yy = df['TARGET_FRAUD']
X_train, X_test, y_train, y_test = train_test_split(xx, yy, test_size=0.33, random_state=seed, stratify=yy)


pd.set_option("display.max_columns", None)
df

,O_EMAIL,O_TELEPHONE,FLAG_DIIDENTICAL,FLAG_NEWSLETTER,CHK_LADR,CHK_RADR,CHK_KTO,CHK_CARD,CHK_COOKIE,CHK_IP,FAIL_LPLZ,FAIL_LORT,FAIL_LPLZORTMATCH,FAIL_RPLZ,FAIL_RORT,FAIL_RPLZORTMATCH,NEW_CUSTOMER,DEL_AMOUNT,DEL_VALUE,SESSION_TIME,DEL_AMOUNT_TOTAL,DEL_VALUE_TOTAL,REM_CUR,REM_MAX,TARGET_FRAUD,P_METHOD_Customer Card,P_METHOD_Direct Debit,P_METHOD_Invoice,P_CARD_TYPE_Amex,P_CARD_TYPE_Eurocard,P_CARD_TYPE_Visa,DEL_DAY_Di,DEL_DAY_Do,DEL_DAY_Fr,DEL_DAY_Mi,DEL_DAY_Mo,DEL_DAY_Sa,DEL_DAY_So,WEEKEND,AVG_ITEM_COST,AVG_ITEM_COST_USUALLY,DISTANCE_USUAL_COST,RECENTLY_ORDERED
Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1,9.8,8,1,33.47,1.0,2.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,True,9.80,33.47,23.67,False
1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1,62.0,9,1,16.64,0.0,1.0,False,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False,62.00,16.64,45.36,False
2,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,1,8.5,5,0,0.00,0.0,0.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,False,8.50,0.00,0.00,False
3,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,1,10.8,5,0,0.00,0.0,0.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,False,10.80,0.00,0.00,False
4,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,1,57.8,22,0,0.00,0.0,0.0,False,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,False,57.80,0.00,0.00,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1,39.5,10,1,55.06,0.0,0.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,True,39.50,55.06,15.56,False
39996,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,1,16.0,10,0,0.00,0.0,0.0,False,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,True,16.00,0.00,0.00,False
39997,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1,41.0,9,1,66.15,0.0,2.0,False,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,False,41.00,66.15,25.15,False


# **Hyperparameter tuning**

## Feature Selection

In [ ]:
'''
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

step_size=1

cv = StratifiedKFold(n_splits=n_splits)
model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])


rfecv = RFECV(estimator=model, step=step_size, cv=cv, scoring='accuracy', importance_getter='named_steps.classifier.feature_importances_')
X_selected_cv = rfecv.fit_transform(X_train, y_train)

print("Recommended amount of features:", rfecv.n_features_)

feature_names = X_train.columns
rankings = rfecv.ranking_
feature_ranking_df = pd.DataFrame({"Feature": feature_names, "Ranking": rankings})
feature_ranking_df = feature_ranking_df.sort_values(by="Ranking")
print(feature_ranking_df)
'''

Although removing the worst-performing features is generally recommended, I found that doing so only decreased accuracy. As a result, I chose not to remove any features based on this results.

However, it is great to see that four of my added features rank among the top 3 :)

## Parameter Tuning:

I am using a voting model that combines multiple different models. Therefore, I have tuned the parameters for each individual model.

In [ ]:
#!pip install optuna

In [ ]:
'''
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

def objective_rfc(trial):

    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 1, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    class_weight = trial.suggest_categorical('class_weight', [None, "balanced"])

    model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=seed,
        n_jobs=-1
        ))
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()



def objective_lg(trial):

    C = trial.suggest_float('C', 1e-4, 1e2, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs', 'saga'])
    class_weight = trial.suggest_categorical('class_weight', [None, "balanced"])

    if solver == 'lbfgs':
        penalty = 'l2'
    else:
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=C,
            solver=solver,
            class_weight=class_weight,
            random_state=seed,
            max_iter=1000,
            penalty=penalty,
            n_jobs=-1
        ))
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()



def objective_gbc(trial):

    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    max_depth = trial.suggest_int('max_depth', 2, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            subsample=subsample,
            max_features=max_features,
            random_state=seed
        ))
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()



def objective_dtc(trial):

    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    max_depth = trial.suggest_int('max_depth', 1, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    splitter = trial.suggest_categorical('splitter', ['best', 'random'])
    class_weight = trial.suggest_categorical('class_weight', [None, "balanced"])

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            splitter=splitter,
            class_weight=class_weight,
            random_state=seed
        ))
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()
    '''

RandomForestClassifier:

In [ ]:
'''
study = optuna.create_study(direction='maximize')
study.optimize(objective_rfc, n_trials=200)

print("Best Parameters:", study.best_params)
'''
# {'n_estimators': 26, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 4, 'class_weight': None, random_state=seed, n_jobs=-1} with value: 0.9417537313432837.

LogisticRegression:

In [ ]:
'''
study = optuna.create_study(direction='maximize')
study.optimize(objective_lg, n_trials=150)

print("Best Parameters:", study.best_params)
'''
# {'C': 0.06508084423841876, 'solver': 'lbfgs', 'class_weight': None, random_state=seed, max_iter=1000, 'penalty': 'l2', n_jobs=-1}. with value: 0.941865671641791.

GradientBoostingClassifier:

In [ ]:
'''
study = optuna.create_study(direction='maximize')
study.optimize(objective_gbc, n_trials=200)

print("Best Parameters:", study.best_params)
'''
# {'n_estimators': 144, 'learning_rate': 0.0706019579462404, 'max_depth': 3, 'min_samples_split': 9, 'min_samples_leaf': 2, 'subsample': 0.9374256739674214, 'max_features': 'sqrt', random_state=seed}. with value: 0.9423880597014925.

DecisionTreeClassifier:

In [ ]:
'''
study = optuna.create_study(direction='maximize')
study.optimize(objective_dtc, n_trials=150)

print("Best Parameters:", study.best_params)
'''
# {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': None, 'splitter': 'best', 'class_weight': None, random_state=seed}. with value: 0.9417164179104478.

# **Training my Model:**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preparing models with hyperparameter-tuned parameters for use in a voting model
dtc_model = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_split=4, min_samples_leaf=2,
                                   max_features=None, splitter='best', class_weight=None, random_state=seed)
gbc_model = GradientBoostingClassifier(n_estimators=144, learning_rate=0.0706019579462404, max_depth=3,
                                       min_samples_split=9, min_samples_leaf=2, subsample=0.9374256739674214,
                                       max_features='sqrt', random_state=seed)
rfc_model = RandomForestClassifier(n_estimators=26, max_depth=15, min_samples_split=2, min_samples_leaf=4,
                                   class_weight=None, random_state=seed, n_jobs=-1)
lg_model = LogisticRegression(C=0.06508084423841876, solver='lbfgs', class_weight=None, random_state=seed, max_iter=1000,
                              penalty='l2', n_jobs=-1)

# My actual model
voting_model = VotingClassifier(estimators=[('DTModel', dtc_model), ('GBCModule', gbc_model), ('RFCModule', rfc_model),
                                            ('LGnModel', lg_model)], voting='soft')
voting_model.fit(X_train_scaled, y_train)

print(f"Train voting model score: {voting_model.score(X_train_scaled, y_train):.4f}")
print(f"Test voting model score: {voting_model.score(X_test_scaled, y_test):.4f}")
print()
print(f"Train ROC score: {roc_auc_score(y_train, voting_model.predict_proba(X_train_scaled)[:, 1]):.4f}")
print(f"Test ROC score: {roc_auc_score(y_test, voting_model.predict_proba(X_test_scaled)[:, 1]):.4f}")

def create_prediction_proba(test_set):

    test_set = scaler.transform(test_set)
    return voting_model.predict_proba(test_set)

Train voting model score: 0.9421
Test voting model score: 0.9414

Train ROC score: 0.8358
Test ROC score: 0.7434


# **Result: Using model on test.csv**

In [ ]:
xx_test = preprocess_df(reload_test_df())
probabilities = create_prediction_proba(xx_test)

submission_df = pd.DataFrame({
    "Id": xx_test.index,
    "TARGET_FRAUD": probabilities[:, 1]
})

submission_df.to_csv("final_test_prediction.csv", index=False)
submission_df

,Id,TARGET_FRAUD
0,40000,0.045383
1,40001,0.099552
2,40002,0.024336
3,40003,0.108593
4,40004,0.021518
...,...,...
9995,49995,0.067554
9996,49996,0.017162
9997,49997,0.095070
9998,49998,0.018866
